# Basic PEFT Type: IA3 (TensorFlow)
**Date**: 2026-05-31  
**Objective**: Basic IA3 implementation from scratch in TensorFlow for support-ticket urgency classification.

## Common Logic Highlight (Shared with PyTorch counterpart)
- Same runtime policy: load configs/runtime.env and honor USE_GPU toggle.
- Same dataset, label mapping, and train/test split logic.
- Same evaluation metrics: accuracy and macro-F1.
- Same PEFT goal: keep base representation mostly frozen and train lightweight adaptation parameters.
- Type-specific basic focus: multiplicative scaling vectors over hidden representations.

In [ ]:
import os
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

In [ ]:
def load_runtime_env() -> None:
    candidate_paths = [
        os.path.join(os.getcwd(), 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), 'configs', 'runtime.env.example'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env'),
        os.path.join(os.getcwd(), '..', 'configs', 'runtime.env.example'),
    ]
    env_loaded = False
    for path in candidate_paths:
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as handle:
            for raw_line in handle:
                line = raw_line.strip()
                if not line or line.startswith('#') or '=' not in line:
                    continue
                key, value = line.split('=', 1)
                os.environ[key.strip()] = value.strip().strip("\"'")
        env_loaded = True
        break
    if not env_loaded and 'USE_GPU' not in os.environ:
        os.environ['USE_GPU'] = '1'

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {'0', 'false', 'no', 'off'}

load_runtime_env()
USE_GPU = parse_use_gpu_flag(os.getenv('USE_GPU', '1'))
GPU_AVAILABLE = len(tf.config.list_physical_devices('GPU')) > 0
RUNTIME_DEVICE = 'gpu' if USE_GPU and GPU_AVAILABLE else 'cpu'
print(f'USE_GPU={int(USE_GPU)} | runtime_device={RUNTIME_DEVICE}')

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    max_tokens: int = 2048
    seq_len: int = 24
    emb_dim: int = 48
    hidden_dim: int = 64
    adapter_rank: int = 8
    batch_size: int = 4
    epochs: int = 2
    lr: float = 1e-3
    train_size: float = 0.8

cfg = ExperimentConfig()

In [ ]:
samples = [
    ('Server not reachable after deployment', 2),
    ('Password reset email not received', 1),
    ('Dashboard typo in heading', 0),
    ('Payment API timing out for premium users', 2),
    ('Need help changing profile picture', 0),
    ('CPU usage spikes to 100 percent hourly', 2),
    ('Can we export reports to CSV?', 0),
    ('Intermittent login failures for SSO users', 2),
    ('Dark mode icon is slightly misaligned', 0),
    ('Data sync lag observed in EU region', 1),
    ('Mobile app crashes on checkout page', 2),
    ('Feature request: bulk archive tickets', 0),
    ('Webhook retries causing duplicate events', 1),
    ('Fraud alert queue delayed by 5 minutes', 2),
    ('Question about invoice date format', 0),
    ('Latency increased after model update', 1),
]
df = pd.DataFrame(samples, columns=['text', 'label'])
df['label_name'] = df['label'].map({0: 'low', 1: 'medium', 2: 'high'})
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split_idx = int(len(df) * cfg.train_size)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=cfg.max_tokens,
    output_mode='int',
    output_sequence_length=cfg.seq_len,
)
vectorizer.adapt(train_df['text'].values)
x_train = vectorizer(train_df['text'].values)
x_test = vectorizer(test_df['text'].values)
y_train = train_df['label'].to_numpy(dtype=np.int32)
y_test = test_df['label'].to_numpy(dtype=np.int32)

In [ ]:
class BasicAdapter(tf.keras.layers.Layer):
    def __init__(self, hidden_dim: int, rank: int, **kwargs):
        super().__init__(**kwargs)
        self.down = tf.keras.layers.Dense(rank, use_bias=False)
        self.up = tf.keras.layers.Dense(hidden_dim, use_bias=False)
    def call(self, inputs, training=False):
        return inputs + self.up(self.down(inputs))

inputs = tf.keras.Input(shape=(cfg.seq_len,), dtype=tf.int64)
embedding = tf.keras.layers.Embedding(cfg.max_tokens, cfg.emb_dim, name='base_embedding')
pool = tf.keras.layers.GlobalAveragePooling1D(name='base_pool')
base_dense = tf.keras.layers.Dense(cfg.hidden_dim, activation='relu', name='base_dense')
x = embedding(inputs)
x = pool(x)
base = base_dense(x)
embedding.trainable = False
base_dense.trainable = False
adapter = BasicAdapter(cfg.hidden_dim, cfg.adapter_rank, name='adapter')(base)
outputs = tf.keras.layers.Dense(3, activation='softmax', name='classifier')(adapter)
model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=cfg.epochs, batch_size=cfg.batch_size, verbose=0)
pred_probs = model.predict(x_test, verbose=0)
pred_labels = pred_probs.argmax(axis=1)
metrics = {
    'accuracy': float(accuracy_score(y_test, pred_labels)),
    'macro_f1': float(f1_score(y_test, pred_labels, average='macro')),
}
print(metrics)

In [ ]:
metric_names = ['accuracy', 'macro_f1']
metric_values = [metrics.get(k, 0.0) for k in metric_names]
plt.figure(figsize=(6, 4))
plt.bar(metric_names, metric_values)
plt.ylim(0.0, 1.0)
plt.title('Basic IA3 TensorFlow: Evaluation Snapshot')
plt.ylabel('Score')
plt.show()

## Summary
- Framework: TensorFlow
- Basic PEFT type: IA3
- Type behavior: multiplicative scaling vectors over hidden representations.
- Common logic is aligned with the matching PyTorch notebook.